# 00. 데이터 다운로드 (AI Hub → Google Drive)

> ⚠️ **이 노트북은 CPU 런타임으로 실행하세요.** (런타임 → 런타임 유형 변경 → CPU)
> 다운로드·압축 해제에는 GPU가 필요 없고, T4 사용량만 소모됩니다.

- AI Hub 데이터는 라이선스상 **저장소에 올리면 안 됩니다.** Drive(`/content/drive/MyDrive/mmdl/data`)에만 저장합니다.
- API 키는 `getpass`로 입력받아 노트북에 저장되지 않습니다.

In [ ]:
# [설정] 경로와 다운로드 대상 — 여기만 수정하면 됩니다
DATA_DIR = "/content/drive/MyDrive/mmdl/data"
RAW_DIR = f"{DATA_DIR}/raw"            # aihubshell 다운로드 위치 (데이터셋별 하위 폴더)
EXTRACT_DIR = f"{DATA_DIR}/extracted"  # 압축 해제 위치

# datasetkey / filekey: AI Hub 데이터셋 페이지 또는 아래 '파일 목록 조회' 셀로 확인
# filekey는 쉼표로 여러 개 입력 가능 (예: "12345,12346")
DOWNLOADS = {
    "emotion":    {"datasetkey": "", "filekey": ""},  # 감정 분류를 위한 대화 음성 데이터셋 - 5차년도_2차 (csv + wav zip)
    "dialect_gs": {"datasetkey": "", "filekey": ""},  # 중·노년층 한국어 방언 - 강원·경상 (2인발화 원천 wav + 라벨링 json)
    "dialect_cc": {"datasetkey": "", "filekey": ""},  # 중·노년층 한국어 방언 - 충청·전라·제주 (2인발화 원천 wav + 라벨링 json)
}

# 압축 해제 시 이 확장자만 풀기 (예: {".json", ".csv"}). None이면 전부.
# wav 수만 개를 Drive에 풀면 매우 느리므로, 탐색 단계에서는 라벨만 먼저 푸는 것을 권장
EXTRACT_EXTS = None
ZIP_NAME_ENCODING = "cp949"  # UTF-8 플래그 없는 zip의 파일명 인코딩 (AI Hub는 대부분 cp949)

In [ ]:
# Google Drive 마운트 후 저장 폴더 생성
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
for d in (RAW_DIR, EXTRACT_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)
print(DATA_DIR)

In [ ]:
# AI Hub API 다운로드 도구(aihubshell) 설치
!curl -s -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x /usr/local/bin/aihubshell
!ls -l /usr/local/bin/aihubshell

In [ ]:
# datasetkey별 파일 목록 조회 (filekey 확인용, API 키 불필요)
import subprocess
for name, d in DOWNLOADS.items():
    if d["datasetkey"]:
        print(f"===== {name} (datasetkey={d['datasetkey']}) =====")
        print(subprocess.run(["aihubshell", "-mode", "l", "-datasetkey", d["datasetkey"]],
                             capture_output=True, text=True).stdout)
    else:
        print(f"[skip] {name}: datasetkey 비어 있음")

In [ ]:
# AI Hub API 키 입력 (getpass → 노트북 파일·출력에 남지 않음)
from getpass import getpass
AIHUB_API_KEY = getpass("AI Hub API 키: ").strip()
print("입력됨" if AIHUB_API_KEY else "비어 있음")

In [ ]:
# 데이터셋별 다운로드 (출력에서 API 키는 *** 로 가림)
import subprocess
assert AIHUB_API_KEY, "API 키를 먼저 입력하세요"
for name, d in DOWNLOADS.items():
    if not (d["datasetkey"] and d["filekey"]):
        print(f"[skip] {name}: datasetkey/filekey 비어 있음")
        continue
    out = Path(RAW_DIR) / name
    out.mkdir(parents=True, exist_ok=True)
    cmd = ["aihubshell", "-mode", "d", "-datasetkey", d["datasetkey"],
           "-filekey", d["filekey"], "-aihubapikey", AIHUB_API_KEY]
    print(f"===== {name} → {out} =====")
    p = subprocess.Popen(cmd, cwd=out, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line.replace(AIHUB_API_KEY, "***"), end="")
    print(f"{name}: {'완료' if p.wait() == 0 else f'실패 (exit {p.returncode})'}")

In [ ]:
# 다운로드된 zip 압축 해제 (RAW_DIR 구조를 EXTRACT_DIR에 그대로, 이미 푼 zip은 건너뜀)
import zipfile
for zp in sorted(Path(RAW_DIR).rglob("*.zip")):
    dest = Path(EXTRACT_DIR) / zp.relative_to(RAW_DIR).with_suffix("")
    done = dest / ".extracted"
    if done.exists():
        print("[skip]", zp.name)
        continue
    with zipfile.ZipFile(zp, metadata_encoding=ZIP_NAME_ENCODING) as zf:
        members = [m for m in zf.infolist()
                   if not EXTRACT_EXTS or Path(m.filename).suffix.lower() in EXTRACT_EXTS]
        print(f"{zp.name}: {len(members)}개 → {dest}")
        zf.extractall(dest, members=members)
    if not EXTRACT_EXTS:  # 일부만 푼 경우엔 완료 표시를 남기지 않음 (나중에 전체 해제 가능)
        done.touch()

In [ ]:
# 결과 확인: 데이터셋 폴더별 확장자 개수와 용량
from collections import Counter
for top in sorted(Path(EXTRACT_DIR).iterdir()):
    print(top.name, dict(Counter(p.suffix.lower() for p in top.rglob("*") if p.is_file())))
!du -sh {RAW_DIR}/* {EXTRACT_DIR}/* 2>/dev/null